# 知识图谱实验: NetworkX图结构与实体关系

> **对应模块**: `src/knowledge_graph.py` (数据挖掘层)
> **前置条件**: `pip install networkx matplotlib scikit-learn`

## 什么是知识图谱？

知识图谱是一种**用图结构表示实体和关系**的数据模型:
- **节点 (Node)**: 代表实体（人、组织、概念等）
- **边 (Edge)**: 代表实体间的关系
- 例如: `Google --owns--> DeepMind --created--> AlphaGo`

在智析项目中，知识图谱从NLP识别的实体中自动构建，
用于展示文档中实体之间的关联，帮助用户理解文档结构。

In [ ]:
import sys
sys.path.insert(0, '..')
from src.knowledge_graph import KnowledgeGraphBuilder, cluster_documents
import os
os.makedirs('../data/processed', exist_ok=True)

# ========================================
# 1.1 手动构建知识图谱
# ========================================

kg = KnowledgeGraphBuilder(graph_type='directed')

# 添加实体节点
entities = [
    ("Google", "ORG"), ("DeepMind", "ORG"), ("AlphaGo", "PRODUCT"),
    ("Demis Hassabis", "PER"), ("London", "LOC"),
    ("Neural Network", "TECH"), ("Reinforcement Learning", "TECH"),
    ("OpenAI", "ORG"), ("GPT-4", "PRODUCT"), ("Sam Altman", "PER"),
]
kg.add_entities(entities)
print(f"添加了 {kg.graph.number_of_nodes()} 个实体节点")

# 添加关系边
relations = [
    ("Google", "owns", "DeepMind"),
    ("DeepMind", "created", "AlphaGo"),
    ("Demis Hassabis", "leads", "DeepMind"),
    ("DeepMind", "located_in", "London"),
    ("AlphaGo", "uses", "Neural Network"),
    ("AlphaGo", "uses", "Reinforcement Learning"),
    ("OpenAI", "created", "GPT-4"),
    ("Sam Altman", "leads", "OpenAI"),
    ("GPT-4", "uses", "Neural Network"),
]
for src, rel, tgt in relations:
    kg.add_relation(src, rel, tgt)
print(f"添加了 {kg.graph.number_of_edges()} 条关系")

In [ ]:
# ========================================
# 1.2 图谱统计信息
# ========================================

stats = kg.get_stats()
print(f"=== 图谱统计 ===")
print(f"节点数: {stats.node_count}")
print(f"边数: {stats.edge_count}")
print(f"\n实体类型分布:")
for etype, count in stats.entity_types.items():
    print(f"  {etype}: {count}个")

print(f"\n度最高的节点 (最'重要'的实体):")
for node_info in stats.top_nodes:
    print(f"  {node_info['node']:>25s}  度={node_info['degree']}  类型={node_info['type']}")

In [ ]:
# ========================================
# 1.3 路径查找 — 发现实体间的隐含关系
# ========================================

print("=== 路径查找 ===\n")

# 查找 Google 到 Neural Network 的路径
paths = kg.find_paths("Google", "Neural Network")
print(f"Google → Neural Network 的路径:")
for path in paths:
    print(f"  {' → '.join(path)}")

# 查找 Google 到 GPT-4 的路径 (可能没有直接路径)
paths = kg.find_paths("Google", "GPT-4")
if paths:
    print(f"\nGoogle → GPT-4 的路径:")
    for path in paths:
        print(f"  {' → '.join(path)}")
else:
    print(f"\nGoogle → GPT-4: 无直接路径 (它们属于不同的子图)")

In [ ]:
# ========================================
# 1.4 知识图谱可视化
# ========================================

print("生成知识图谱可视化...")
path = kg.visualize('../data/processed/knowledge_graph.png')

if path:
    from PIL import Image
    import matplotlib.pyplot as plt
    
    img = Image.open(path)
    plt.figure(figsize=(14, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('AI领域知识图谱')
    plt.tight_layout()
    plt.show()
    print(f"图谱已保存: {path}")

In [ ]:
# ========================================
# 1.5 从文本自动构建共现关系
# ========================================

text = """
Google acquired DeepMind in 2014. Demis Hassabis leads DeepMind in London.
DeepMind created AlphaGo which uses neural networks and reinforcement learning.
OpenAI, founded by Sam Altman, created GPT-4 using neural networks.
Google and OpenAI are competitors in the AI field.
"""

kg2 = KnowledgeGraphBuilder()
entities_auto = [
    ("Google", "ORG"), ("DeepMind", "ORG"), ("Demis Hassabis", "PER"),
    ("London", "LOC"), ("AlphaGo", "PRODUCT"), ("neural networks", "TECH"),
    ("reinforcement learning", "TECH"), ("OpenAI", "ORG"),
    ("Sam Altman", "PER"), ("GPT-4", "PRODUCT"),
]
kg2.add_entities(entities_auto)
kg2.add_relations_from_text(text, entities_auto)

stats2 = kg2.get_stats()
print(f"自动构建: {stats2.node_count}节点, {stats2.edge_count}条关系")
print(f"\n自动发现的关系:")
for edge in kg2.graph.edges(data=True):
    print(f"  {edge[0]} --{edge[2].get('relation', '?')}--> {edge[1]}")

In [ ]:
# ========================================
# 1.6 主题聚类 (cluster_documents)
# ========================================

texts = [
    "机器学习是人工智能的核心分支，包括监督学习和无监督学习。",
    "深度学习使用神经网络进行特征学习和模式识别。",
    "自然语言处理让计算机能够理解和生成人类语言。",
    "计算机视觉使机器能够从图像和视频中提取信息。",
    "强化学习通过奖励信号让智能体学习最优策略。",
    "Python是数据科学和机器学习最常用的编程语言。",
    "Pandas和NumPy是Python数据分析的基础库。",
    "Matplotlib和Seaborn用于数据可视化和探索性分析。",
]

result = cluster_documents(texts, n_clusters=3)

print(f"=== 主题聚类结果 ({result['n_clusters']}个簇) ===\n")
for cluster_id in range(result['n_clusters']):
    texts_in_cluster = [texts[i] for i, l in enumerate(result['labels']) if l == cluster_id]
    print(f"簇 {cluster_id}:")
    print(f"  关键词: {', '.join(result['top_words'][cluster_id][:5])}")
    for t in texts_in_cluster:
        print(f"  - {t[:50]}...")
    print()

---
## 总结

### KnowledgeGraphBuilder 核心方法:
| 方法 | 功能 |
|------|------|
| `add_entities()` | 批量添加实体节点 |
| `add_relation()` | 添加关系边 |
| `add_relations_from_text()` | 从文本自动提取共现关系 |
| `build_from_nlp_result()` | 从NLP结果构建图谱 |
| `get_stats()` | 获取统计信息 |
| `find_paths()` | 查找实体间路径 |
| `get_subgraph()` | 获取子图 |
| `visualize()` | 可视化图谱 |
| `save()`/`load()` | 保存和加载 |

### 下一步: 运行 `06_streamlit_app.ipynb` 学习Web应用部署